In [1]:
# Data format
# import datetime
from data_tmp import UsageEvent


from base_bloom import BloomFilter
from base_hll import HyperLogLog
# from base_reservoir import
# from base_map_reduce import
 

from typing import List, Dict, Set, Optional, Tuple
from datetime import datetime, timedelta
import mmh3
from collections import defaultdict
import itertools
import math
import random
from bitarray import bitarray


In [2]:
UsageEvent

data_tmp.UsageEvent

In [3]:
class StreamingAnalyser:
    def __init__(self,
                 hll_precision: int = 14,
                 bloom_size: int = 100000,
                 bloom_hash_count: int = 5,
                 time_window: timedelta = timedelta(hours=1)):
        
        # Ініціалізація компонентів
        # Унікальні глядачі
        self.track_listeners = defaultdict(lambda: HyperLogLog(hll_precision))
        # Повторні перегляди
        self.repeat_listens  = BloomFilter(bloom_size, bloom_hash_count)
        
        # Статистика прослуховувань
        self.listen_counts = defaultdict(int)
        self.completion_stats = defaultdict(lambda: defaultdict(int))
        self.device_stats = defaultdict(lambda: defaultdict(int))
        
        # Аналіз плейлистів та рекомендацій
        self.user_sequences = defaultdict(list)
        self.track_correlations = defaultdict(lambda: defaultdict(int))
        
        # Часові метрики
        self.time_window = time_window
        self.window_stats = []
        self.hourly_stats = defaultdict(lambda: defaultdict(int))
        
        # Додаткова статистика
        self.duration_points = defaultdict(list)

    def proccesor(self, event):
        """Обробляє подію прослуховування."""
        # Конвертуємо timestamp в datetime
        event_time = datetime.strptime(event.timestamp, '%Y-%m-%d %H:%M:%S')
        
        # Оновлюємо HyperLogLog
        self.track_listeners[event.music_id].add(event.user_id)
        
        # Перевіряємо повторне прослуховування
        listen_key = f"{event.user_id}:{event.music_id}:{event_time.date()}"
        is_repeat = self.repeat_listens.check(listen_key)
        self.repeat_listens.add(listen_key)
        
        # Оновлюємо всі метрики
        self._update_listen_stats(event)
        self._update_device_stats(event)
        self._update_user_sequences(event)
        self._update_time_window(event, event_time)
        self._update_duration_stats(event)


  
    def _update_listen_stats(self, event: UsageEvent) -> None:
        """Оновлює статистику прослуховувань."""
        self.listen_counts[event.music_id] += 1
        
        if event.completed.lower() == 'true':
            self.completion_stats[event.music_id]['completed'] += 1
        self.completion_stats[event.music_id]['total'] += 1
    
    def _update_device_stats(self, event: UsageEvent) -> None:
        """Оновлює статистику пристроїв."""
        self.device_stats[event.music_id][event.device] += 1
    
    def _update_user_sequences(self, event: UsageEvent) -> None:
        """Оновлює послідовності прослуховувань для рекомендацій."""
        user_seq = self.user_sequences[event.user_id]
        if user_seq:
            last_track = user_seq[-1]
            self.track_correlations[last_track][event.music_id] += 1
        
        user_seq.append(event.music_id)
        if len(user_seq) > 10:  # Зберігаємо останні 10 треків
            user_seq.pop(0)
    
    def _update_time_window(self, event: UsageEvent, event_time: datetime) -> None:
        """Підтримує ковзне вікно для аналізу трендів."""
        self.window_stats = [
            (ts, track) for ts, track in self.window_stats 
            if event_time - ts <= self.time_window
        ]
        
        self.window_stats.append((event_time, event.music_id))
        self.hourly_stats[event.music_id][event_time.hour] += 1
    
    def _update_duration_stats(self, event: UsageEvent) -> None:
        """Оновлює статистику тривалості прослуховування."""
        try:
            duration = float(event.duration_point)
            self.duration_points[event.music_id].append(duration)
        except (ValueError, TypeError):
            pass  # Пропускаємо некоректні значення
    
    def get_trending_tracks(self, top_n: int = 10) -> List[Tuple[str, int]]:
        """Повертає трендові треки на основі поточного вікна."""
        window_counts = defaultdict(int)
        for _, track_id in self.window_stats:
            window_counts[track_id] += 1
        
        return sorted(
            window_counts.items(),
            key=lambda x: x[1],
            reverse=True
        )[:top_n]
    
    def get_track_recommendations(self, track_id: str, limit: int = 5) -> List[Tuple[str, float]]:
        """Повертає рекомендації на основі кореляцій прослуховувань."""
        correlations = self.track_correlations[track_id]
        if not correlations:
            return []
        
        total_plays = sum(correlations.values())
        normalized = [
            (track, count / total_plays)
            for track, count in correlations.items()
        ]
        
        return sorted(normalized, key=lambda x: x[1], reverse=True)[:limit]
    
    def get_track_stats(self, track_id: str) -> Dict[str, any]:
        """Повертає детальну статистику для треку."""
        completion_stats = self.completion_stats[track_id]
        duration_points = self.duration_points[track_id]
        
        return {
            'total_plays': self.listen_counts[track_id],
            'unique_listeners': int(self.track_listeners[track_id].estimate()),
            'avg_duration': sum(duration_points) / len(duration_points) if duration_points else 0,
            'completion_rate': (
                completion_stats['completed'] / completion_stats['total']
                if completion_stats['total'] > 0 else 0
            ),
            'device_distribution': {
                device: count / self.listen_counts[track_id]
                for device, count in self.device_stats[track_id].items()
            },
            'hourly_distribution': dict(self.hourly_stats[track_id])
        }

    

In [4]:
import random
import string
import time
from dataclasses import dataclass
from typing import Generator

def random_string(length: int) -> str:
    """Generate a random string of uppercase letters and digits."""
    return ''.join(random.choices(string.ascii_uppercase + string.digits, k=length))

def generate_usage_events():
    """A generator that yields random UsageEvent objects."""
    devices = ["mobile", "desktop", "tablet", "smart_tv"]
    track_ids = [f"track_{i}" for i in range(100)]
    user_ids = [f"user_{i}" for i in range(1000)]

    while True:
        user_id = random.choice(user_ids)
        music_id = random.choice(track_ids)
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        duration_point = str(random.randint(30, 300))  # Duration in seconds (30 to 300 seconds)
        completed = random.choice(["true", "false"])
        device = random.choice(devices)

        yield UsageEvent(
            user_id=user_id,
            music_id=music_id,
            timestamp=timestamp,
            duration_point=duration_point,
            completed=completed,
            device=device
        )

In [5]:
analizer = StreamingAnalyser()
event_generator = generate_usage_events()


for _ in range(10000):  # Generate 5 events for demonstration
    event = next(event_generator)
    print(event)
    analizer.proccesor(event)

UsageEvent(user_id='user_233', music_id='track_65', timestamp='2025-07-05 13:19:29', duration_point='256', completed='true', device='tablet')
UsageEvent(user_id='user_580', music_id='track_0', timestamp='2025-07-05 13:19:29', duration_point='72', completed='true', device='smart_tv')
UsageEvent(user_id='user_680', music_id='track_51', timestamp='2025-07-05 13:19:29', duration_point='163', completed='false', device='tablet')
UsageEvent(user_id='user_483', music_id='track_99', timestamp='2025-07-05 13:19:29', duration_point='69', completed='true', device='tablet')
UsageEvent(user_id='user_906', music_id='track_53', timestamp='2025-07-05 13:19:29', duration_point='72', completed='false', device='smart_tv')
UsageEvent(user_id='user_169', music_id='track_97', timestamp='2025-07-05 13:19:29', duration_point='272', completed='true', device='smart_tv')
UsageEvent(user_id='user_315', music_id='track_18', timestamp='2025-07-05 13:19:29', duration_point='163', completed='false', device='desktop')


In [6]:
for track_id, plays in analizer.get_trending_tracks():
    print(f"\nТрек {track_id}")

        # Детальна статистика
    stats = analizer.get_track_stats(track_id)
    print(f"Загальні прослуховування: {stats['total_plays']}")
    print(f"Унікальні слухачі: {stats['unique_listeners']}")
    print(f"Середня тривалість прослуховування: {stats['avg_duration']:.2f}")
    print(f"Відсоток завершених прослуховувань: {stats['completion_rate']:.1%}")
    
    # Розподіл за пристроями
    print("\nРозподіл за пристроями:")
    for device, percentage in stats['device_distribution'].items():
        print(f"  - {device}: {percentage:.1%}")
    
    # Рекомендації
    print("\nРекомендовані треки:")
    for rec_id, score in analizer.get_track_recommendations(track_id):
        print(f"  - {rec_id} (score: {score:.2f})")


Трек track_0
Загальні прослуховування: 127
Унікальні слухачі: 62
Середня тривалість прослуховування: 167.07
Відсоток завершених прослуховувань: 56.7%

Розподіл за пристроями:
  - smart_tv: 25.2%
  - mobile: 27.6%
  - desktop: 30.7%
  - tablet: 16.5%

Рекомендовані треки:
  - track_25 (score: 0.04)
  - track_10 (score: 0.04)
  - track_58 (score: 0.03)
  - track_31 (score: 0.03)
  - track_50 (score: 0.03)

Трек track_22
Загальні прослуховування: 121
Унікальні слухачі: 50
Середня тривалість прослуховування: 154.73
Відсоток завершених прослуховувань: 41.3%

Розподіл за пристроями:
  - desktop: 26.4%
  - smart_tv: 24.8%
  - mobile: 19.8%
  - tablet: 28.9%

Рекомендовані треки:
  - track_53 (score: 0.04)
  - track_72 (score: 0.03)
  - track_84 (score: 0.03)
  - track_32 (score: 0.03)
  - track_69 (score: 0.03)

Трек track_69
Загальні прослуховування: 117
Унікальні слухачі: 59
Середня тривалість прослуховування: 162.60
Відсоток завершених прослуховувань: 54.7%

Розподіл за пристроями:
  - de

## Комплексне завдання: Аналіз хештегів у соціальних мережах 

Гаразд, давай розберемо це завдання крок за кроком. Ми реалізуємо відсутні частини (HyperLogLog та BloomFilter) у спрощеному вигляді, щоб було зрозуміло, як вони працюють, а потім пояснимо весь код.

**Мета:** Створити систему, яка аналізує пости з соціальних мереж для виявлення трендів хештегів, унікальних користувачів та вірусних хештегів, при цьому ефективно використовуючи пам'ять.



**Крок 1: Розуміння основних структур даних**

Перш ніж перейти до основного аналізатора, нам потрібні дві важливі структури даних для оптимізації:

1.  **HyperLogLog (HLL):** Алгоритм для приблизного підрахунку кількості унікальних елементів у великому наборі даних. Він використовує значно менше пам'яті, ніж точний підрахунок (наприклад, зберігання всіх унікальних ID в `set`).
2.  **Bloom Filter:** Імовірнісна структура даних, яка дозволяє швидко перевірити, чи може елемент бути членом набору. Можливі хибнопозитивні спрацювання (каже, що елемент є, хоча його немає), але хибнонегативних немає (якщо каже, що немає, то його точно немає). Використовується для економії ресурсів при перевірках.



**Крок 2: Реалізація HyperLogLog (спрощена)**

Ми не будемо реалізовувати повний HLL з усіма оптимізаціями, а створимо базову версію, щоб показати принцип.


In [7]:

import mmh3 # Потрібен для хешування, встановіть: pip install mmh3
import math
from typing import List, Dict, Set, Optional, Tuple



In [8]:

class HyperLogLog:
    def __init__(self, precision: int = 14):
        """
        precision (p): визначає кількість 'регістрів' (m = 2^p).
        Чим вище p, тим точніший підрахунок, але більше пам'яті.
        """
        self.p = precision
        self.m = 1 << precision  # m = 2^p
        self.registers = [0] * self.m
        # Константа для корекції, залежить від m
        if self.m == 16: self.alpha = 0.673
        elif self.m == 32: self.alpha = 0.697
        elif self.m == 64: self.alpha = 0.709
        else: self.alpha = 0.7213 / (1 + 1.079 / self.m)

    def _get_register_index_and_rank(self, value_str: str) -> (int, int):
        """
        Хешує значення, визначає індекс регістра та ранг (кількість нулів на початку).
        """
        hash_val = mmh3.hash(value_str, seed=42) # 32-бітний хеш

        # Перші p біт для індексу регістра
        register_index = hash_val & (self.m - 1) # Еквівалент hash_val % self.m, якщо m - ступінь 2

        # Решта біт (32-p) для підрахунку нулів на початку
        # Зсуваємо вправо на p біт, щоб отримати частину для рангу
        rank_bits = hash_val >> self.p
        
        # Підрахунок нулів на початку (rank)
        # Ми шукаємо позицію першої одиниці в бітах, що залишилися
        # (після зсуву на p). Максимальна кількість біт для рангу - це 32-p.
        rank = 0
        if rank_bits == 0: # Якщо всі біти - нулі (малоймовірно, але можливо)
            rank = 32 - self.p # Максимальний ранг
        else:
            # Знаходимо позицію першої '1' справа наліво
            # Наприклад, ...01000 -> rank = 3 + 1 = 4
            # Це кількість послідовних нулів + 1
            while (rank_bits & 1) == 0 and rank < (32 - self.p):
                rank += 1
                rank_bits >>= 1
            rank += 1 # Сам ранг - це позиція першої '1'
        return register_index, rank

    def add(self, value_str: str) -> None:
        """Додає елемент до HLL."""
        register_index, rank = self._get_register_index_and_rank(value_str)
        self.registers[register_index] = max(self.registers[register_index], rank)

    def estimate(self) -> float:
        """Оцінює кількість унікальних елементів."""
        sum_inv_powers_of_2 = 0.0
        for val in self.registers:
            sum_inv_powers_of_2 += 2.0**(-val)
        
        estimate = self.alpha * (self.m**2) / sum_inv_powers_of_2
        
        # Корекція для малих та великих значень
        num_zero_registers = self.registers.count(0)
        if estimate <= 2.5 * self.m: # Малі значення
            if num_zero_registers > 0:
                estimate = self.m * math.log(self.m / num_zero_registers) # LinearCounting
        elif estimate > (1/30.0) * (2**32): # Великі значення (тут 2**32 - максимальне значення 32-бітного хешу)
             estimate = -(2**32) * math.log(1.0 - estimate / (2**32))
             
        return estimate



**Крок 3: Реалізація Bloom Filter (спрощена)**


In [9]:

class BloomFilter:
    def __init__(self, size: int = 100000, hash_count: int = 5):
        """
        size: розмір бітового масиву.
        hash_count: кількість хеш-функцій.
        """
        self.size = size
        self.hash_count = hash_count
        self.bit_array = [False] * size # Використовуємо список булевих значень для простоти

    def _get_hashes(self, item_str: str) -> List[int]:
        """Генерує 'hash_count' хешів для елемента."""
        hashes = []
        # Використовуємо mmh3 з різними 'seed' для отримання різних хешів
        for i in range(self.hash_count):
            hashes.append(mmh3.hash(item_str, seed=i) % self.size)
        return hashes

    def add(self, item_str: str) -> None:
        """Додає елемент до фільтра."""
        for h_val in self._get_hashes(item_str):
            self.bit_array[h_val] = True

    def check(self, item_str: str) -> bool:
        """
        Перевіряє, чи може елемент бути в фільтрі.
        True: елемент МОЖЕ бути в наборі (можливе хибнопозитивне спрацювання).
        False: елемента ТОЧНО НЕМАЄ в наборі.
        """
        for h_val in self._get_hashes(item_str):
            if not self.bit_array[h_val]:
                return False
        return True




**Крок 4: Розбір класу `HashtagAnalyzer`**

Тепер, коли у нас є `HyperLogLog` та `BloomFilter`, ми можемо детально розглянути `HashtagAnalyzer`.



In [10]:

from dataclasses import dataclass
from typing import List, Dict, Set, Optional, Tuple
import datetime # Використовуємо datetime з модуля datetime
from collections import defaultdict
import itertools
# mmh3, HyperLogLog, BloomFilter вже визначені вище

@dataclass
class Post:
    user_id: str
    timestamp: datetime.datetime # Уточнено тип
    hashtags: List[str]
    engagement: int  # лайки + репости

class HashtagAnalyzer:
    def __init__(self,
                 hll_precision: int = 14,    # Точність для HLL (кількість регістрів = 2^14)
                 bloom_size: int = 1000000, # Розмір бітового масиву для фільтра Блума
                 bloom_hash_count: int = 7): # Кількість хеш-функцій для фільтра Блума
        
        # 1. Відстеження унікальних користувачів для кожного хештегу
        # defaultdict(lambda: ...) створює новий HyperLogLog, якщо ключ (хештег) ще не існує
        self.hashtag_users = defaultdict(lambda: HyperLogLog(hll_precision))
        
        # 3. Аналіз "вірусності" - фільтр Блума для швидкої перевірки, чи бачили ми комбінацію "хештег:дата"
        self.viral_detector = BloomFilter(bloom_size, bloom_hash_count)
        
        # 2. Виявлення трендових комбінацій хештегів (пар)
        self.hashtag_pairs = defaultdict(int) # Лічильник для пар хештегів
        
        # Для аналізу трендів у ковзному вікні (наприклад, за останню годину)
        self.timestamp_window_posts = [] # Зберігатимемо (timestamp, hashtags) з постів
        self.current_trends = {} # Для відстеження швидкості поширення (кандидати на вірусність)

    def process_post(self, post: Post) -> None:
        """Обробляє один пост."""
        
        # --- 1. Унікальні користувачі ---
        for hashtag in post.hashtags:
            # Додаємо user_id до HLL відповідного хештегу
            self.hashtag_users[hashtag].add(post.user_id)
            
            # --- 3. Аналіз "вірусності" ---
            # Ключ для фільтра Блума: "хештег:дата_поста"
            # Це дозволяє нам швидко перевірити, чи це "нова" згадка хештегу СЬОГОДНІ.
            # Якщо хештег згадується вперше за день, це може бути початком вірусного поширення.
            # (Можна використовувати точніший час, наприклад, годину, але дата простіша для демонстрації)
            key_for_virality_check = f"{hashtag}:{post.timestamp.date()}"
            
            # Якщо фільтр Блума каже, що ми, МОЖЛИВО, НЕ бачили цю комбінацію
            # (або бачили, але це хибнопозитивне спрацювання, що рідко,
            # або дійсно не бачили)
            if not self.viral_detector.check(key_for_virality_check):
                self.viral_detector.add(key_for_virality_check) # Додаємо до фільтра
                self._update_viral_trends(hashtag, post.timestamp) # Оновлюємо дані для вірусності

        # --- 2. Трендові комбінації хештегів ---
        # Обробляємо тільки якщо в пості більше одного хештегу
        if len(post.hashtags) > 1:
            self._process_hashtag_pairs(post.hashtags)
        
        # --- Оновлення часового вікна для аналізу трендів (не використовується активно в get_viral_hashtags, але є основою) ---
        self._update_time_window(post)
    
    def _process_hashtag_pairs(self, hashtags: List[str]) -> None:
        """Аналізує та підраховує популярні комбінації (пари) хештегів."""
        # Сортуємо хештеги, щоб пара (a,b) була такою ж, як (b,a)
        # itertools.combinations генерує всі унікальні пари
        for pair in itertools.combinations(sorted(hashtags), 2):
            self.hashtag_pairs[pair] += 1
    
    def _update_viral_trends(self, hashtag: str, timestamp: datetime.datetime) -> None:
        """
        Відстежує потенційно вірусні хештеги.
        Записує час першої "нової" згадки (завдяки фільтру Блума) та кількість таких згадок.
        """
        if hashtag not in self.current_trends:
            self.current_trends[hashtag] = {
                'first_seen_today': timestamp, # Час, коли ми вперше зафіксували цей хештег сьогодні (завдяки Bloom Filter)
                'mentions_today': 1            # Кількість "перших згадок за день"
            }
        else:
            # Якщо хештег знову з'явився як "новий за день" (можливо, помилка Bloom Filter або новий день),
            # оновлюємо час, якщо він пізніший, і збільшуємо лічильник.
            # В ідеалі, ця логіка має бути складнішою для справжньої вірусності,
            # але для демо це показує ідею.
            self.current_trends[hashtag]['mentions_today'] += 1
            # Можна оновлювати 'first_seen_today', якщо timestamp раніший, але це залежить від логіки.
            # Для простоти, ми фіксуємо першу згадку.
            # Якщо ми хочемо відстежувати швидкість зростання за КОРОТКИЙ період,
            # то `current_trends` мав би містити детальнішу історію згадок (наприклад, список timestamp'ів).

    def _update_time_window(self, post: Post) -> None:
        """
        Підтримує ковзне вікно постів (наприклад, за останню годину).
        Це потрібно для аналізу трендів у реальному часі.
        """
        window_duration = datetime.timedelta(hours=1) # Тривалість вікна - 1 година
        current_time = post.timestamp # Час поточного поста
        
        # Видаляємо старі записи (ті, що вийшли за межі вікна)
        # self.timestamp_window_posts зберігає (timestamp, list_of_hashtags)
        self.timestamp_window_posts = [
            (ts, tags) for ts, tags in self.timestamp_window_posts
            if current_time - ts <= window_duration
        ]
        
        # Додаємо новий запис (час та хештеги поточного поста)
        self.timestamp_window_posts.append((post.timestamp, post.hashtags))

    # --- Методи для отримання результатів аналізу ---

    def get_unique_users(self, hashtag: str) -> int:
        """Повертає оцінку кількості унікальних користувачів для заданого хештегу."""
        if hashtag in self.hashtag_users:
            return int(self.hashtag_users[hashtag].estimate())
        return 0 # Якщо хештегу не було
    
    def get_trending_pairs(self, top_n: int = 10) -> List[Tuple[Tuple[str, str], int]]:
        """Повертає N найпопулярніших пар хештегів."""
        return sorted(
            self.hashtag_pairs.items(), # .items() дає список (ключ, значення)
            key=lambda item: item[1],   # Сортуємо за значенням (кількістю)
            reverse=True                # У порядку спадання
        )[:top_n]                       # Беремо перші N елементів
    
    def get_viral_hashtags(self, viral_mentions_threshold: int = 5, viral_time_window_hours: int = 1) -> Dict[str, dict]:
        """
        Повертає хештеги, які демонструють ознаки вірусності.
        Логіка вірусності тут спрощена: якщо хештег мав кілька "перших згадок за день"
        (зафіксованих через _update_viral_trends) протягом останньої години.
        """
        viral_tags = {}
        # Важливо: для реальної вірусності потрібен більш складний аналіз, що враховує швидкість росту
        # згадок за короткий проміжок часу, порівняно зі звичайною активністю.
        # Тут ми спростимо: якщо 'first_seen_today' було нещодавно і 'mentions_today' достатньо велике.
        
        # Поточний час для порівняння (можна передавати як параметр для тестування)
        # У реальній системі це буде datetime.datetime.now(datetime.timezone.utc)
        # Для демонстрації використаємо час останнього обробленого поста, якщо є, або поточний час.
        if self.timestamp_window_posts:
            # Якщо є оброблені пости, беремо час останнього
            # Це важливо, бо generate_test_data генерує пости в минулому.
            # datetime.datetime.now() тут дасть неправильні результати для "вірусності" згенерованих даних.
            current_processing_time = self.timestamp_window_posts[-1][0] 
        else:
            current_processing_time = datetime.datetime.now(datetime.timezone.utc if datetime.datetime.now().tzinfo else None)


        analysis_window = datetime.timedelta(hours=viral_time_window_hours)

        for tag, data in self.current_trends.items():
            time_since_first_seen_today = current_processing_time - data['first_seen_today']
            
            # Чи була перша "нова" згадка нещодавно?
            if time_since_first_seen_today <= analysis_window:
                # Чи достатньо "нових" згадок?
                if data['mentions_today'] >= viral_mentions_threshold:
                    # Розрахуємо приблизну швидкість (згадок / хвилину) за цей період
                    # Це дуже груба оцінка, бо 'mentions_today' це не загальна кількість згадок.
                    duration_seconds = time_since_first_seen_today.total_seconds()
                    if duration_seconds == 0: duration_seconds = 60 # Уникаємо ділення на нуль, припускаємо 1 хв
                    
                    mentions_per_minute = data['mentions_today'] / (duration_seconds / 60.0)
                    
                    viral_tags[tag] = {
                        'first_seen_in_window': data['first_seen_today'].isoformat(),
                        'distinct_daily_mentions_in_window': data['mentions_today'],
                        'approx_mentions_per_minute': mentions_per_minute
                    }
        
        # Сортуємо за 'approx_mentions_per_minute' для кращого вигляду
        return dict(sorted(viral_tags.items(), key=lambda item: item[1]['approx_mentions_per_minute'], reverse=True))




**Крок 5: Генерація тестових даних та запуск аналізу**

Функція `generate_test_data` створює список об'єктів `Post` для тестування.
Функція `analyze_social_media_posts` ініціалізує `HashtagAnalyzer`, обробляє пости та виводить результати.



In [11]:

# Приклад використання
def analyze_social_media_posts(posts: List[Post], analyzer: HashtagAnalyzer) -> None: # Передаємо аналізатор
    # Обробка постів
    for i, post in enumerate(posts):
        analyzer.process_post(post)
        if (i + 1) % 1000 == 0: # Логування прогресу
             print(f"Processed {i+1}/{len(posts)} posts...")
    
    print("\n--- Результати Аналізу Хештегів ---")
    
    # 1. Унікальні користувачі за деякими хештегами
    print("\n1. Оцінка унікальних користувачів за хештегами:")
    # Виберемо декілька хештегів, які точно є в даних
    test_tags_for_users = list(analyzer.hashtag_users.keys())[:5] # Перші 5 хештегів, що зустрілися
    if not test_tags_for_users and posts: # Якщо ще немає ключів, але є пости, візьмемо з останнього
        test_tags_for_users = posts[-1].hashtags
        
    for tag in test_tags_for_users:
        users = analyzer.get_unique_users(tag)
        print(f"#{tag}: ~{users} унікальних користувачів")
    
    # 2. Найпопулярніші пари хештегів
    print("\n2. Топ-10 популярних пар хештегів:")
    for (tag1, tag2), count in analyzer.get_trending_pairs(top_n=10):
        print(f"  #{tag1} + #{tag2}: {count} разів")
    
    # 3. Вірусні хештеги (за нашою спрощеною логікою)
    print("\n3. Потенційно вірусні хештеги (за останню годину обробки):")
    viral_hashtags = analyzer.get_viral_hashtags(viral_mentions_threshold=3, viral_time_window_hours=1)
    if not viral_hashtags:
        print("  Вірусних хештегів не виявлено за поточними критеріями.")
    for tag, stats in viral_hashtags.items():
        print(f"  #{tag}:")
        print(f"    - Перша згадка (в вікні): {stats['first_seen_in_window']}")
        print(f"    - \"Нових\" згадок за день (в вікні): {stats['distinct_daily_mentions_in_window']}")
        print(f"    - ~{stats['approx_mentions_per_minute']:.2f} \"нових\" згадок/хв")


In [12]:

# Генерація тестових даних
def generate_test_data(num_posts: int = 10000) -> List[Post]:
    import random # Імпортуємо тут, щоб не було глобальним
    
    # Більше різноманітних хештегів
    hashtags_pool = ['python', 'coding', 'tech', 'ai', 'ml', 'data', 'datascience',
                     'programming', 'developer', 'analytics', 'cloud', 'webdev',
                     'innovation', 'startup', 'marketing', 'socialmedia', 'news',
                     'crypto', 'blockchain', 'future', 'learning', 'education']
    
    # Генеруємо пости протягом останніх 3 годин, щоб було що аналізувати у вікні
    base_time = datetime.datetime.now(datetime.timezone.utc if datetime.datetime.now().tzinfo else None) - datetime.timedelta(hours=3)
    
    posts = []
    for i in range(num_posts):
        # Випадковий набір хештегів (від 1 до 4)
        num_tags_in_post = random.randint(1, 4)
        post_tags = random.sample(hashtags_pool, num_tags_in_post)
        
        # Генеруємо час посту в межах останніх 3 годин
        time_offset = datetime.timedelta(
            minutes=random.randint(0, 3 * 60) # Від 0 до 180 хвилин
        )
        post_timestamp = base_time + time_offset
        
        post = Post(
            user_id=f"user_{random.randint(1, num_posts // 10)}", # Менше унікальних юзерів для HLL
            timestamp=post_timestamp,
            hashtags=post_tags,
            engagement=random.randint(0, 1000)
        )
        posts.append(post)
    
    # Сортуємо пости за часом, це важливо для правильної роботи ковзного вікна
    posts.sort(key=lambda p: p.timestamp)
    return posts


In [13]:

# === Запуск аналізу ===
if __name__ == "__main__":
    print("Генерація тестових даних...")
    # Зменшимо кількість постів для швидшого демо, але можна збільшити до 10000-100000
    test_posts = generate_test_data(num_posts=20000) 
    print(f"Згенеровано {len(test_posts)} постів.")
    
    print("\Ініціалізація аналізатора...")
    # Параметри можна налаштовувати. 
    # Для HLL: p=10 (1024 регістри) - швидше, менш точно. p=14 (16384 регістри) - точніше, повільніше.
    # Для Bloom: size залежить від очікуваної кількості унікальних елементів (ключів "хештег:дата").
    #            hash_count зазвичай 5-10.
    analyzer_instance = HashtagAnalyzer(hll_precision=12, bloom_size=500000, bloom_hash_count=5)
    
    print("Обробка постів...")
    analyze_social_media_posts(test_posts, analyzer_instance)
    print("\n--- Аналіз завершено ---")




Генерація тестових даних...
Згенеровано 20000 постів.
\Ініціалізація аналізатора...
Обробка постів...
Processed 1000/20000 posts...
Processed 2000/20000 posts...
Processed 3000/20000 posts...
Processed 4000/20000 posts...
Processed 5000/20000 posts...
Processed 6000/20000 posts...
Processed 7000/20000 posts...
Processed 8000/20000 posts...
Processed 9000/20000 posts...
Processed 10000/20000 posts...
Processed 11000/20000 posts...
Processed 12000/20000 posts...
Processed 13000/20000 posts...
Processed 14000/20000 posts...
Processed 15000/20000 posts...
Processed 16000/20000 posts...
Processed 17000/20000 posts...
Processed 18000/20000 posts...
Processed 19000/20000 posts...
Processed 20000/20000 posts...

--- Результати Аналізу Хештегів ---

1. Оцінка унікальних користувачів за хештегами:
#datascience: ~1386 унікальних користувачів
#ai: ~1334 унікальних користувачів
#startup: ~1363 унікальних користувачів
#tech: ~1352 унікальних користувачів
#webdev: ~1341 унікальних користувачів

2. То



**Крок 6: Пояснення рішення**





### Використані алгоритми та структури даних:

1.  **HyperLogLog (HLL)**
    *   **Призначення:** Відстеження *приблизної* кількості унікальних користувачів для кожного хештегу (`self.hashtag_users`).
    *   **Чому:** Значно економить пам'ять порівняно з точним підрахунком (наприклад, `set` з ID користувачів), що критично при мільйонах постів і тисячах хештегів. Похибка підрахунку зазвичай невелика (кілька відсотків).
    *   **Як працює (спрощено):** Хешує ID користувача. Частина хешу використовується для вибору "регістра", інша частина – для визначення "рангу" (наприклад, кількість нулів на початку). У регістрі зберігається максимальний побачений ранг. Оцінка кількості унікальних елементів базується на гармонійному середньому значень у регістрах.

2.  **Фільтр Блума (Bloom Filter)**
    *   **Призначення:** Швидка перевірка, чи з'являлася комбінація "хештег + дата" раніше (`self.viral_detector`). Використовується для виявлення "перших" згадок хештегу за день, що може сигналізувати про початок вірусного поширення.
    *   **Чому:** Дуже швидка перевірка та додавання, низьке використання пам'яті. Дозволяє уникнути дорогих запитів до бази даних або повного сканування для перевірки "новизни".
    *   **Як працює (спрощено):** Елемент хешується декількома різними хеш-функціями. Результати хешування використовуються як індекси у бітовому масиві, де відповідні біти встановлюються в '1'. Для перевірки, чи є елемент, знову хешуємо його і дивимося, чи всі відповідні біти встановлені. Якщо хоч один біт '0' – елемента точно немає. Якщо всі '1' – елемент, *ймовірно*, є (можливі хибнопозитивні спрацювання).

3.  **`defaultdict` та лічильники**
    *   **Призначення:** `self.hashtag_pairs = defaultdict(int)` використовується для підрахунку частоти появи пар хештегів. `defaultdict` зручний тим, що автоматично створює новий лічильник (з значенням 0 для `int`) при зверненні до неіснуючого ключа.
    *   **Чому:** Просто та ефективно для підрахунку.

4.  **`itertools.combinations`**
    *   **Призначення:** Ефективно генерує всі унікальні пари хештегів з одного поста (`_process_hashtag_pairs`).
    *   **Чому:** Оптимізована стандартна бібліотека для комбінаторних операцій.



### Оптимізації та особливості:

1.  **Ковзне вікно (`_update_time_window`)**
    *   **Призначення:** `self.timestamp_window_posts` зберігає інформацію про пости (час, хештеги) за останній визначений період (наприклад, 1 година). Це дозволяє аналізувати тренди в реальному часі, фокусуючись на найсвіжіших даних.
    *   **Чому:** Обмежує обсяг даних для аналізу, роблячи його швидшим та менш вимогливим до пам'яті, ніж аналіз всієї історії. Дозволяє виявляти короткострокові тренди.

2.  **Аналіз комбінацій хештегів (`_process_hashtag_pairs`)**
    *   **Особливість:** Сортування хештегів перед створенням пари (`sorted(hashtags)`) гарантує, що пара `(#python, #coding)` буде такою ж, як `(#coding, #python)`, що важливо для коректного підрахунку.

3.  **Метрики вірусності (`_update_viral_trends`, `get_viral_hashtags`)**
    *   **Підхід:** Вірусність тут визначається спрощено:
        *   Фільтр Блума допомагає ідентифікувати "першу згадку хештегу за день".
        *   `self.current_trends` накопичує інформацію про такі "перші згадки".
        *   `get_viral_hashtags` аналізує, чи було достатньо таких "перших згадок" нещодавно (в межах `viral_time_window_hours`) і чи перевищують вони певний поріг (`viral_mentions_threshold`).
    *   **Чому такий підхід:** Це спрощення. У реальності, вірусність – це швидке зростання частоти згадок за короткий проміжок часу порівняно зі звичайною активністю хештегу. Наша модель фіксує лише "нові сплески" активності протягом дня.



### Можливість розподіленої обробки (теоретично):

Хоча код не реалізує розподілену обробку, використані структури та підходи можуть бути адаптовані:

1.  **MapReduce-подібний підхід:**
    *   **Map фаза:** Пости можуть оброблятися паралельно на різних вузлах. Кожен вузол генерує часткові дані:
        *   Для HLL: кожен вузол може мати свої HLL для хештегів.
        *   Для пар хештегів: кожен вузол рахує свої локальні пари.
        *   Для фільтра Блума: кожен вузол може оновлювати свій локальний фільтр або використовувати спільний (з обережністю).
    *   **Reduce фаза:**
        *   **HLL:** Регістри HLL з різних вузлів можна об'єднати (взяти максимум для кожного індексу регістра) для отримання глобальної оцінки.
        *   **Пари хештегів:** Лічильники пар з різних вузлів просто сумуються.
        *   **Фільтр Блума:** Бітові масиви фільтрів Блума можна об'єднати операцією OR побітово.
2.  **Розділення даних (Sharding):** Пости можна розділяти за часом, за хештегом (якщо є основний), або за ID користувача для розподіленої обробки.



 > Цей приклад демонструє основні концепції та дає базу для розуміння, як можна підійти до аналізу великих обсягів даних з соціальних мереж, використовуючи імовірнісні структури даних для ефективності.

## Комплексне завдання: Аналіз даних стримінгової платформи


**Основна ідея:** Ми будемо обробляти потік подій перегляду (`ViewEvent`). На основі цих подій ми оновлюватимемо різні метрики: загальну кількість переглядів, кількість унікальних глядачів (HLL), послідовності переглядів для рекомендацій, а також відстежуватимемо тренди у ковзному вікні.



**Крок 1: Визначення структури даних для події перегляду**


In [14]:

from dataclasses import dataclass
import datetime

@dataclass
class ViewEvent:
    user_id: str
    video_id: str
    timestamp: datetime.datetime
    watch_duration_seconds: int  # Додамо тривалість перегляду, може бути корисно
    # session_id: Optional[str] = None # Можна додати для більш глибокого аналізу сесій



**Крок 2: Підготовка допоміжних класів (HyperLogLog, BloomFilter)**



Ми використаємо ті самі спрощені реалізації `HyperLogLog` та `BloomFilter` з попереднього завдання. Для реального застосування варто було б взяти готові оптимізовані бібліотеки (наприклад, `datasketch` для HLL та Bloom Filter).



In [15]:

import mmh3 # pip install mmh3
import math
from collections import defaultdict, deque # deque для ефективного ковзного вікна історії користувача
from typing import List, Dict, Tuple, Deque, Any # Додамо Deque




**Крок 3: Розробка основного класу `StreamingAnalyzer`**


In [16]:

class StreamingAnalyzer:
    def __init__(self,
                 hll_precision: int = 12,
                 bloom_size: int = 1_000_000, # Для відстеження (користувач, відео) переглядів
                 bloom_hash_count: int = 7,
                 sliding_window_duration_seconds: int = 3600, # 1 година
                 max_user_history_for_sequence: int = 5): # Довжина послідовності для рекомендацій

        # 1. Підрахунок переглядів (загальний)
        self.video_total_views = defaultdict(int)

        # 2. Відстеження унікальних глядачів для кожного відео
        self.video_unique_viewers_hll = defaultdict(lambda: HyperLogLog(hll_precision))

        # (Опціонально) Фільтр Блума для виявлення нещодавніх переглядів (користувач, відео)
        # Може використовуватись, щоб не враховувати повторні швидкі перегляди в деяких аналітиках
        # або для функціоналу "продовжити перегляд". Для поточних вимог ми не будемо його активно
        # використовувати, але покажемо, як він міг би ініціалізуватися.
        self.user_video_recent_views_bloom = BloomFilter(bloom_size, bloom_hash_count)

        # 3. Аналіз послідовностей переглядів (Граф кореляцій для рекомендацій)
        # video_A_id -> {video_B_id: count, video_C_id: count}
        self.view_sequences_graph = defaultdict(lambda: defaultdict(int))
        # Зберігаємо невелику історію переглядів для кожного користувача
        self.user_watch_history: Dict[str, Deque[str]] = defaultdict(
            lambda: deque(maxlen=max_user_history_for_sequence)
        )

        # 4. Ковзне вікно для аналізу трендів (популярні відео за останній час)
        self.sliding_window_events: Deque[Tuple[datetime.datetime, str]] = deque() # (timestamp, video_id)
        self.sliding_window_duration = datetime.timedelta(seconds=sliding_window_duration_seconds)

    def process_view_event(self, event: ViewEvent) -> None:
        """Обробляє одну подію перегляду."""
        video_id = event.video_id
        user_id = event.user_id
        timestamp = event.timestamp

        # 1. Оновлення загальних переглядів
        self.video_total_views[video_id] += 1

        # 2. Оновлення унікальних глядачів
        self.video_unique_viewers_hll[video_id].add(user_id)

        # (Опціонально) Оновлення фільтра Блума для нещодавніх переглядів
        # bloom_key = f"{user_id}:{video_id}" # Можна додати часову мітку (напр., година)
        # if not self.user_video_recent_views_bloom.check(bloom_key):
        #     self.user_video_recent_views_bloom.add(bloom_key)

        # 3. Оновлення послідовностей переглядів
        user_history = self.user_watch_history[user_id]
        if user_history: # Якщо є попередні перегляди у цього користувача
            previous_video_id = user_history[-1] # Останнє переглянуте відео
            if previous_video_id != video_id: # Не враховуємо перегляд того ж відео поспіль
                self.view_sequences_graph[previous_video_id][video_id] += 1
        user_history.append(video_id) # Додаємо поточне відео до історії

        # 4. Оновлення ковзного вікна для трендів
        self._update_sliding_window(timestamp, video_id)

    def _update_sliding_window(self, current_timestamp: datetime.datetime, video_id: str) -> None:
        """Додає подію до ковзного вікна та видаляє застарілі."""
        # Додаємо нову подію
        self.sliding_window_events.append((current_timestamp, video_id))
        
        # Видаляємо старі події, які вийшли за межі вікна
        # Робимо це, поки найстаріша подія у вікні занадто стара
        while self.sliding_window_events and \
              current_timestamp - self.sliding_window_events[0][0] > self.sliding_window_duration:
            self.sliding_window_events.popleft() # Видаляємо зліва (найстарішу)

    # --- Методи для отримання аналітики ---

    def get_trending_videos(self, top_n: int = 10) -> List[Tuple[str, int]]:
        """Виявляє трендові відео на основі ковзного вікна."""
        if not self.sliding_window_events:
            return []
        
        video_counts_in_window = defaultdict(int)
        for _, video_id in self.sliding_window_events:
            video_counts_in_window[video_id] += 1
            
        sorted_videos = sorted(video_counts_in_window.items(), key=lambda item: item[1], reverse=True)
        return sorted_videos[:top_n]

    def get_unique_viewers_for_video(self, video_id: str) -> int:
        """Повертає оцінку унікальних глядачів для відео."""
        if video_id in self.video_unique_viewers_hll:
            return int(self.video_unique_viewers_hll[video_id].estimate())
        return 0

    def get_content_recommendations(self, video_id: str, top_n: int = 5, normalize: bool = True) -> List[Tuple[str, float]]:
        """
        Генерує рекомендації на основі відео, які часто дивляться після заданого.
        Нормалізація (P(B|A) = count(A,B) / count(A_total_views)) може покращити якість.
        """
        if video_id not in self.view_sequences_graph:
            return []

        candidate_videos = self.view_sequences_graph[video_id]
        
        if not normalize:
            # Проста сортування за кількістю переходів
            sorted_recommendations = sorted(candidate_videos.items(), key=lambda item: item[1], reverse=True)
            return [(vid, float(count)) for vid, count in sorted_recommendations[:top_n]]

        # Нормалізація: P(B|A) = count(A->B) / views(A)
        # views(A) - це загальна кількість разів, коли після A щось дивилися.
        # Або можна використовувати self.video_total_views[video_id] якщо хочемо співвідношення до загальних переглядів A.
        # Для P(B|A) краще використовувати загальну кількість вихідних переходів з А.
        total_transitions_from_video_id = sum(candidate_videos.values())
        if total_transitions_from_video_id == 0:
            return [] # Немає переходів з цього відео

        recommendations_with_score = []
        for next_video_id, count in candidate_videos.items():
            score = count / total_transitions_from_video_id
            recommendations_with_score.append((next_video_id, score))
            
        sorted_recommendations = sorted(recommendations_with_score, key=lambda item: item[1], reverse=True)
        return sorted_recommendations[:top_n]

    def get_videos_for_caching(self, top_n: int = 10, use_trending: bool = True) -> List[Tuple[str, int]]:
        """
        Визначає відео для кешування. Може базуватися на загальній популярності або трендах.
        """
        if use_trending:
            # Трендові відео вже відсортовані за популярністю у вікні
            return self.get_trending_videos(top_n)
        else:
            # На основі загальної кількості переглядів
            sorted_videos = sorted(self.video_total_views.items(), key=lambda item: item[1], reverse=True)
            return sorted_videos[:top_n]

    # --- Методи для можливого розподілення ---
    def merge_analyzer_data(self, other_analyzer: 'StreamingAnalyzer') -> None:
        """
        Об'єднує дані з іншого екземпляра аналізатора.
        Корисно для розподіленої обробки, де кожен вузол обробляє свою частину даних.
        """
        # 1. Загальні перегляди
        for video_id, count in other_analyzer.video_total_views.items():
            self.video_total_views[video_id] += count

        # 2. Унікальні глядачі (HLL)
        for video_id, hll in other_analyzer.video_unique_viewers_hll.items():
            if video_id in self.video_unique_viewers_hll:
                self.video_unique_viewers_hll[video_id].merge(hll)
            else:
                self.video_unique_viewers_hll[video_id] = hll # Копіюємо, якщо не було

        # 3. Граф кореляцій
        for prev_video_id, next_videos in other_analyzer.view_sequences_graph.items():
            for next_video_id, count in next_videos.items():
                self.view_sequences_graph[prev_video_id][next_video_id] += count
        
        # Ковзне вікно та історія користувачів зазвичай не об'єднуються так просто,
        # вони залежать від потоку подій. Для них потрібна інша стратегія
        # (наприклад, централізоване ковзне вікно або пересилання подій).
        # Фільтр Блума теж складніше об'єднувати без втрати точності (OR по бітах).
        print("Warning: Sliding window, user history, and Bloom filters are not merged in this simple example.")




**Крок 4: Генерація тестових даних та запуск аналізу**


In [17]:

import random

def generate_mock_view_events(num_events: int = 10000, num_users: int = 1000, num_videos: int = 200) -> List[ViewEvent]:
    events = []
    user_ids = [f"user_{i}" for i in range(num_users)]
    video_ids = [f"video_{i}" for i in range(num_videos)]
    
    # Зробимо деякі відео більш популярними
    popular_videos = random.sample(video_ids, k=num_videos // 10) # 10% відео - популярні
    
    # Змоделюємо деякі послідовності
    # Користувач, що дивиться video_A, з більшою ймовірністю подивиться video_B
    sequence_pairs = {}
    for _ in range(num_videos // 20): # 5% відео мають наступне "рекомендоване" відео
        source_vid = random.choice(video_ids)
        target_vid = random.choice(video_ids)
        if source_vid != target_vid:
            sequence_pairs[source_vid] = target_vid
            
    # Відстежуємо останнє відео для кожного користувача для симуляції послідовностей
    last_video_watched_by_user = {}

    current_time = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(hours=2) # Починаємо 2 години тому

    for i in range(num_events):
        user_id = random.choice(user_ids)
        
        # Вибір відео
        video_id = None
        # З ймовірністю 30% користувач подивиться відео з послідовності, якщо попереднє було відповідним
        if user_id in last_video_watched_by_user and \
           last_video_watched_by_user[user_id] in sequence_pairs and \
           random.random() < 0.7: # Збільшимо ймовірність переходу по послідовності
            video_id = sequence_pairs[last_video_watched_by_user[user_id]]
        
        if video_id is None:
            # З ймовірністю 60% вибираємо популярне відео, інакше - випадкове
            if random.random() < 0.6:
                video_id = random.choice(popular_videos)
            else:
                video_id = random.choice(video_ids)

        last_video_watched_by_user[user_id] = video_id

        # Генеруємо час посту в межах останніх 2 годин
        time_offset = datetime.timedelta(seconds=random.randint(0, int(2 * 3600 * (i / num_events))))
        event_timestamp = current_time + time_offset
        
        watch_duration = random.randint(30, 600) # від 30 сек до 10 хв
        
        events.append(ViewEvent(user_id, video_id, event_timestamp, watch_duration))
        
    # Важливо: сортуємо події за часом для коректної роботи ковзного вікна та послідовностей
    events.sort(key=lambda e: e.timestamp)
    return events


In [18]:

print("Генерація тестових подій перегляду...")
# Зменшимо кількість для швидкого демо
mock_events = generate_mock_view_events(num_events=100000, num_users=1000, num_videos=500)
print(f"Згенеровано {len(mock_events)} подій.")


Генерація тестових подій перегляду...
Згенеровано 100000 подій.


In [19]:

print("\nІніціалізація аналізатора...")
analyzer = StreamingAnalyzer(
    hll_precision=10, # Менша точність для швидкості, але більша похибка
    sliding_window_duration_seconds=1800, # 30 хвилин вікно
    max_user_history_for_sequence=3
)



Ініціалізація аналізатора...


In [20]:

print("Обробка подій...")
for i, event in enumerate(mock_events):
    analyzer.process_view_event(event)
    if (i + 1) % 10000 == 0:
        print(f"  Оброблено {i+1}/{len(mock_events)} подій...")

print("\n--- Результати Аналізу Стримінгової Платформи ---")


Обробка подій...
  Оброблено 10000/100000 подій...
  Оброблено 20000/100000 подій...
  Оброблено 30000/100000 подій...
  Оброблено 40000/100000 подій...
  Оброблено 50000/100000 подій...
  Оброблено 60000/100000 подій...
  Оброблено 70000/100000 подій...
  Оброблено 80000/100000 подій...
  Оброблено 90000/100000 подій...
  Оброблено 100000/100000 подій...

--- Результати Аналізу Стримінгової Платформи ---


In [21]:

# 1. Трендові відео (за останній період ковзного вікна)
print("\n1. Трендові відео (за останній період):")
trending = analyzer.get_trending_videos(top_n=5)
if trending:
    for video, count in trending:
        print(f"  - {video}: {count} переглядів у вікні")
else:
    print("  Немає даних про тренди (можливо, вікно ще порожнє або замало даних).")



1. Трендові відео (за останній період):
  - video_293: 61 переглядів у вікні
  - video_181: 58 переглядів у вікні
  - video_144: 56 переглядів у вікні
  - video_56: 55 переглядів у вікні
  - video_304: 55 переглядів у вікні


In [22]:

# 2. Унікальні глядачі для деяких відео
print("\n2. Унікальні глядачі для відео:")
videos_to_check = [event.video_id for event in random.sample(mock_events, min(5, len(mock_events)))] if mock_events else []
if trending: # Додамо трендові відео до перевірки
    videos_to_check.extend([v[0] for v in trending])

checked_videos = set()
for video_id in videos_to_check:
    if video_id in checked_videos or not video_id: continue # Уникаємо дублів та None
    unique_viewers = analyzer.get_unique_viewers_for_video(video_id)
    total_views = analyzer.video_total_views.get(video_id, 0)
    print(f"  - {video_id}: ~{unique_viewers} унікальних глядачів (загалом {total_views} переглядів)")
    checked_videos.add(video_id)



2. Унікальні глядачі для відео:
  - video_62: ~89 унікальних глядачів (загалом 92 переглядів)
  - video_330: ~103 унікальних глядачів (загалом 106 переглядів)
  - video_228: ~693 унікальних глядачів (загалом 1273 переглядів)
  - video_318: ~83 унікальних глядачів (загалом 88 переглядів)
  - video_444: ~699 унікальних глядачів (загалом 1306 переглядів)
  - video_293: ~701 унікальних глядачів (загалом 1328 переглядів)
  - video_181: ~695 унікальних глядачів (загалом 1233 переглядів)
  - video_144: ~670 унікальних глядачів (загалом 1252 переглядів)
  - video_56: ~709 унікальних глядачів (загалом 1243 переглядів)
  - video_304: ~711 унікальних глядачів (загалом 1279 переглядів)


In [23]:

# 3. Рекомендації контенту
print("\n3. Рекомендації контенту:")
video_for_reco = None
if trending:
    video_for_reco = trending[0][0] # Беремо найтрендовіше відео
elif videos_to_check:
    video_for_reco = random.choice(list(checked_videos))




3. Рекомендації контенту:


In [ ]:

if video_for_reco:
    print(f"  Рекомендації для '{video_for_reco}':")
    recommendations_raw = analyzer.get_content_recommendations(video_for_reco, top_n=3, normalize=False)
    if recommendations_raw:
        print("    Без нормалізації (за абсолютною кількістю переходів):")
        for video, score in recommendations_raw:
            print(f"      - {video} (raw_count: {int(score)})")
    else:
            print(f"    Немає сирих рекомендацій для {video_for_reco}")

    recommendations_norm = analyzer.get_content_recommendations(video_for_reco, top_n=3, normalize=True)
    if recommendations_norm:
        print("    З нормалізацією (ймовірність P(B|A)):")
        for video, score in recommendations_norm:
            print(f"      - {video} (score: {score:.3f})")
    else:
        print(f"    Немає нормалізованих рекомендацій для {video_for_reco}")

else:
    print("  Не вдалося вибрати відео для демонстрації рекомендацій.")
    


  Рекомендації для 'video_95':
    Без нормалізації (за абсолютною кількістю переходів):
      - video_29 (raw_count: 211)
      - video_20 (raw_count: 207)
      - video_3 (raw_count: 205)
    З нормалізацією (ймовірність P(B|A)):
      - video_29 (score: 0.071)
      - video_20 (score: 0.070)
      - video_3 (score: 0.069)


In [24]:
# 4. Відео для кешування
print("\n4. Топ-5 відео для кешування (на основі трендів):")
caching_videos_trending = analyzer.get_videos_for_caching(top_n=5, use_trending=True)
if caching_videos_trending:
    for video, count in caching_videos_trending:
        print(f"  - {video} (переглядів у вікні: {count})")
else:
    print("  Немає даних для кешування на основі трендів.")
    
print("\n4.1. Топ-5 відео для кешування (на основі загальної популярності):")
caching_videos_total = analyzer.get_videos_for_caching(top_n=5, use_trending=False)
if caching_videos_total:
    for video, count in caching_videos_total:
        print(f"  - {video} (загальна кількість переглядів: {count})")
else:
    print("  Немає даних для кешування на основі загальної популярності.")

print("\n--- Аналіз завершено ---")




4. Топ-5 відео для кешування (на основі трендів):
  - video_293 (переглядів у вікні: 61)
  - video_181 (переглядів у вікні: 58)
  - video_144 (переглядів у вікні: 56)
  - video_56 (переглядів у вікні: 55)
  - video_304 (переглядів у вікні: 55)

4.1. Топ-5 відео для кешування (на основі загальної популярності):
  - video_146 (загальна кількість переглядів: 1354)
  - video_293 (загальна кількість переглядів: 1328)
  - video_445 (загальна кількість переглядів: 1327)
  - video_444 (загальна кількість переглядів: 1306)
  - video_481 (загальна кількість переглядів: 1305)

--- Аналіз завершено ---



**Пояснення ключових моментів реалізації:**

1.  **Підрахунок переглядів (`video_total_views`):** Простий `defaultdict(int)` для підрахунку загальної кількості переглядів кожного відео.
2.  **Унікальні глядачі (`video_unique_viewers_hll`):** `defaultdict(lambda: HyperLogLog(...))` зберігає окремий HLL для кожного `video_id`. Коли користувач дивиться відео, його `user_id` додається до HLL цього відео. Це дає приблизну кількість унікальних глядачів.
3.  **Послідовності переглядів та Рекомендації (`view_sequences_graph`, `user_watch_history`):**
    *   `user_watch_history`: Для кожного користувача зберігається `deque` (двостороння черга з обмеженою максимальною довжиною) останніх переглянутих `video_id`. `deque` ефективно додає та видаляє елементи з обох кінців.
    *   `view_sequences_graph`: Це `defaultdict(lambda: defaultdict(int))`, що представляє зважений орієнтований граф. Ключ – `video_id`, з якого йде перехід. Значення – словник, де ключ – `video_id`, на який переходять, а значення – кількість таких переходів.
    *   При обробці події, якщо користувач вже щось дивився, ми беремо попереднє відео з його історії та збільшуємо лічильник переходу `previous_video_id -> current_video_id`.
    *   **Нормалізація рекомендацій:** Замість простої кількості переходів, ми можемо розрахувати умовну ймовірність P(B|A) = "кількість переходів A->B" / "загальна кількість переходів з A". Це часто дає більш релевантні рекомендації, оскільки враховує загальну популярність відео А.
4.  **Трендові відео (`sliding_window_events`):**
    *   `sliding_window_events`: `deque`, що зберігає пари `(timestamp, video_id)` для подій, які відбулися протягом останнього `sliding_window_duration`.
    *   При додаванні нової події старі події, що вийшли за межі вікна, видаляються з початку `deque`.
    *   `get_trending_videos` просто підраховує частоту `video_id` у поточному вікні.
5.  **Оптимізація кешування (`get_videos_for_caching`):** Може базуватися або на поточних трендах (відео, популярні *зараз*), або на загальній популярності відео за весь час. Вибір залежить від стратегії кешування.
6.  **Фільтр Блума (`user_video_recent_views_bloom`):** У цьому прикладі він ініціалізується, але активно не використовується для основних метрик. Його можна було б задіяти, наприклад, для:
    *   Ідентифікації "дійсно нових" переглядів (якщо користувач не дивився це відео протягом, скажімо, останньої доби).
    *   Уникнення врахування швидких повторних завантажень/відкриттів відео як окремих сесій для аналізу послідовностей.
7.  **Можливість розподіленої обробки:**
    *   Метод `merge_analyzer_data` показує, як дані з різних екземплярів аналізатора (які могли б працювати на різних вузлах, обробляючи різні частини потоку подій) можуть бути об'єднані.
    *   HLL легко об'єднуються (поелементний максимум регістрів).
    *   Лічильники (загальні перегляди, граф кореляцій) просто сумуються.
    *   Об'єднання ковзних вікон та фільтрів Блума складніше і вимагає більш продуманих стратегій.
    *   **Шардинг:** Дані можна було б шардувати за `user_id` (кожен вузол обробляє свій набір користувачів, добре для аналізу поведінки) або `video_id` (добре для агрегації статистики по відео).



> Цей приклад дає гарну основу. У реальній системі кожен компонент був би, ймовірно, окремим сервісом або частиною більшої архітектури обробки даних (наприклад, з використанням Apache Kafka для потоків подій, Apache Spark/Flink для розподіленої обробки, та спеціалізованих баз даних/сховищ для результатів).